# Rainfall Pattern & Crop Suitability Analyser

The model classes are in `src/rainfall.py`; analysis and data acquisition are in `src/rainfall_analysis.py`. The three supplied region profiles are illustrative monthly totals, not observed climatological records. Broad expected rainfall-zone context is compared with the World Bank Uganda historical climate portal [2].

## Crop threshold sources

The FAO [Irrigation Water Management Training Manual 3, Chapter 3](https://www.fao.org/4/s2022e/s2022e07.htm) gives indicative growing-period water needs of 500–800 mm for maize and 300–500 mm for beans, together with typical growing-period lengths [1]. The monthly screening thresholds are derived by dividing the seasonal minimum/maximum by the crop-duration maximum/minimum, respectively: maize 83–192 mm/month; dry beans 82–158 mm/month. These are rainfall proxies, not planting prescriptions: crop water need is evapotranspiration, and effective rainfall depends on soil, timing, variety, and local climate.

For coffee, the same FAO manual recommends a year-round crop coefficient near 0.95 [1]. Applying that to a screening reference ET range of 3–6 mm/day and a 30-day month gives an illustrative 86–171 mm/month proxy. Thresholds should be calibrated with local agronomy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine as scipy_cosine_distance

from src.rainfall import CropRule, Region
from src.rainfall_analysis import (
    FAO_CROP_WATER_REFERENCE,
    WORLD_BANK_UGANDA_CLIMATE_REFERENCE,
    analyze_rainy_seasons,
    classify_months,
    cosine_similarity,
    create_crop_rules,
    create_regions,
    farmer_advisory,
    pairwise_region_metrics,
    plot_nasa_monthly_boxplots,
    run_analysis,
    run_real_data_extension,
)

In [ ]:
results = run_analysis()
results['region_summary'].round(1)

In [ ]:
crop_rules = results['crop_rules']
pd.DataFrame([{
    'Crop': rule.crop_name,
    'Minimum monthly rainfall (mm)': rule.minimum_monthly_mm,
    'Maximum monthly rainfall (mm)': rule.maximum_monthly_mm,
    'Threshold basis': rule.source,
} for rule in crop_rules]).set_index('Crop')

In [ ]:
results['classifications'].pivot_table(
    index=['region', 'crop'], columns='month', values='classification',
    aggfunc='first',
).reindex(columns=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

## Correct cosine similarity

`math.cos()` takes one angle in radians; it does not compare rainfall vectors. Vector cosine similarity is the dot product divided by the product of Euclidean norms. SciPy documents cosine distance as one minus this normalized dot product [4]. Cosine similarity measures profile shape and ignores positive scale: if one region's monthly rainfall is a constant multiple of another's, cosine similarity is 1 even when its total rainfall is much higher. Pearson correlation centers each profile and measures co-variation; Euclidean distance retains the absolute mm differences.

In [ ]:
for measure, matrix in results['similarities'].items():
    print(measure.replace('_', ' ').title())
    print(matrix.round(3).to_string())

In [ ]:
print('Seasonal interpretation reference:', WORLD_BANK_UGANDA_CLIMATE_REFERENCE)
results['rainy_seasons']

In [ ]:
print(results['advisory'])
assert len(results['advisory'].split()) <= 200

In [ ]:
rain_figure, rain_axis = plt.subplots(figsize=(13, 10))
rain_axis.imshow(plt.imread(results['plot_path']))
rain_axis.axis('off')
rain_figure

## Ten-year NASA POWER extension

NASA POWER's [monthly point API](https://power.larc.nasa.gov/docs/services/api/temporal/monthly/) supplies corrected precipitation (`PRECTOTCORR`) in mm/day [3]. The extension multiplies each monthly daily mean by that calendar month's days, drops the API's annual summary key, saves ten years for all three coordinates, and makes one box plot per region to show year-to-year variability by month. This gridded product is not a local rain gauge; compare with station data where available.

In [ ]:
historical_csv = Path('data/nasa_power_rainfall_2015_2024.csv')
historical_plot = Path('outputs/nasa_monthly_rainfall_boxplots.png')
if historical_csv.exists():
    historical_rainfall = pd.read_csv(historical_csv)
else:
    extension = run_real_data_extension(2015, 2024)
    historical_rainfall = extension['data']

if not historical_plot.exists():
    plot_nasa_monthly_boxplots(historical_rainfall, historical_plot)
print(historical_rainfall.groupby('region')['year'].nunique())
historical_rainfall.head()

In [ ]:
real_monthly_normals = historical_rainfall.groupby(
    ['region', 'month_number'], as_index=False
)[['rainfall_mm']].mean()
real_rainfall_data = {
    region_name: group.sort_values('month_number')['rainfall_mm'].to_numpy()
    for region_name, group in real_monthly_normals.groupby('region')
}
real_regions = create_regions(real_rainfall_data)
analyze_rainy_seasons(real_regions)

In [ ]:
nasa_figure, nasa_axis = plt.subplots(figsize=(12, 10))
nasa_axis.imshow(plt.imread(historical_plot))
nasa_axis.axis('off')
nasa_figure

## Findings & Limitations

The illustrative profiles produce annual totals of 1,600 mm for Kampala, 1,388 mm for Gulu, and 1,040 mm for Mbarara. Cosine similarity makes Kampala and Mbarara look relatively alike (0.891), although their totals differ by 560 mm; it captures seasonal shape more than rainfall volume. Pearson correlation tells a different story: Kampala–Gulu is slightly negative (-0.066), consistent with their peaks occurring at different times. No single similarity score answers both “same timing?” and “same amount?”

The illustrative peak detector identifies Kampala and Mbarara as bimodal and Gulu as unimodal, matching the expected broad zones. It uses SciPy `find_peaks` with chosen prominence and separation settings [5]. However, applying it to NASA POWER 2015–2024 monthly normals classifies Gulu as bimodal too. This disagreement shows that peak count depends on data source, averaging period, prominence, and separation settings; it should not be presented as a definitive climate-zone label without station validation. NASA annual totals also vary materially across the decade, so one climatological profile hides substantial year-to-year risk.

Crop labels are screening prompts, not planting recommendations. The monthly cutoffs approximate seasonal evapotranspiration requirements, but rainfall is not effective soil water: runoff, dry spells within a month, soil storage, crop variety, and planting date matter. NASA POWER is gridded model/reanalysis data, not a farm rain gauge. Use local station observations, soil conditions, and extension advice before acting on the suggested maize months.

## References

1. Brouwer, C., & Heibloem, M. (1986). *Irrigation Water Management: Irrigation Water Needs*, Training Manual No. 3, Chapter 3. FAO. https://www.fao.org/4/s2022e/s2022e07.htm. (Indicative crop water needs, durations, and crop coefficients; monthly cutoffs here are derived screening proxies.)
2. World Bank Group. *Climate Change Knowledge Portal: Uganda Historical Climate*. https://climateknowledgeportal.worldbank.org/country/uganda/climate-data-historical. (Broad climatology context; not used as the source of the illustrative vectors.)
3. NASA Langley Research Center. *POWER Monthly and Annual API*. https://power.larc.nasa.gov/docs/services/api/temporal/monthly/. (Source for the optional 2015–2024 `PRECTOTCORR` download.)
4. SciPy Developers. `scipy.spatial.distance.cosine` API reference. https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cosine.html.
5. SciPy Developers. `scipy.signal.find_peaks` API reference. https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html.